## IMPORTS

In [2]:
import requests as r
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

## FUNCIONES

In [3]:
def abrir_Web_WTA(url):
    driver = webdriver.Chrome()
    driver.get(url)
    driver.implicitly_wait(10)
    aceptar_cookies(driver)
    time.sleep(5)
    cerrar_popup_weblayer(driver, "weblayer--box-web-push-subscription")
    time.sleep(1.5)
    cerrar_popup_weblayer(driver, "weblayer--box-cookieless")
    return driver

In [4]:
def aceptar_cookies(driver):
    driver.implicitly_wait(10)
    try:
        aceptar = driver.find_element(By.ID, "onetrust-accept-btn-handler")
        aceptar.click()
    except Exception as e:
        print("No se encontró el botón de aceptar cookies:", e)

In [5]:
# Como tenemos que hacer click en dos diferentes y ambos en shadowbox, vamos a crear funcion
def cerrar_popup_weblayer(driver, nombre_clase_popup):
    try:
        # 1. Encontrar el elemento "host" (el div que contiene el Shadow Root)
        # Iteramos sobre los divs porque el host suele no tener ID o clase visible
        host_elements = driver.find_elements(By.TAG_NAME, "div")
        host = None
        
        for element in host_elements:
            try:
                # Intentamos acceder al shadow root
                shadow = element.shadow_root
                # Verificamos si dentro está el popup que buscamos
                if shadow.find_elements(By.CSS_SELECTOR, f".{nombre_clase_popup}"):
                    host = element
                    break
            except:
                # Si el div no tiene shadow root, lanza excepción y continuamos
                continue
        
        if not host:
            print(f"⚠️ No se encontró el popup: {nombre_clase_popup}")
            return False

        # 2. Acceder al Shadow Root
        shadow_root = host.shadow_root

        # 3. Buscar el botón de cerrar (X) y hacer clic
        # Usamos aria-label="Close" porque es el estándar en ambos popups
        boton_close = shadow_root.find_element(By.CSS_SELECTOR, "button[aria-label='Close']")
        boton_close.click()
        
        print(f"✅ Popup '{nombre_clase_popup}' cerrado exitosamente.")
        return True

    except Exception as e:
        print(f"❌ Error al intentar cerrar el popup '{nombre_clase_popup}': {e}")
        return False

## CODIGO

### Almacenar torneos

In [12]:
url = "https://wtatennis.com/tournaments"

In [13]:
driver = abrir_Web_WTA(url)

✅ Popup 'weblayer--box-web-push-subscription' cerrado exitosamente.
✅ Popup 'weblayer--box-cookieless' cerrado exitosamente.


In [15]:
try:
    cambio_all = driver.find_element(By.CLASS_NAME, "tournament-list__status-filter")
    cambio_all.click()
except Exception as e:
    print("No se pudo hacer clic en el filtro 'All':", e)

In [16]:
# Obtener los nombres del torneo y su enlace
tournaments = {}
try:
    torneos = driver.find_elements(By.CLASS_NAME, "tournament-list__absolute-link")

    for torneo in torneos:
        nombre_torneo = torneo.get_attribute("title")
        enlace_torneo = torneo.get_attribute("href")

        tournaments[nombre_torneo] = enlace_torneo

except Exception as e:
    print("Error al obtener los torneos:", e)

driver.close()

In [17]:
for torneo, enlace in tournaments.items():
    print(f"Torneo: {torneo}, Enlace: {enlace}")

Torneo: United Cup, Enlace: https://www.wtatennis.com/tournaments/united-cup
Torneo: Brisbane International, Enlace: https://www.wtatennis.com/tournaments/800/brisbane/2026
Torneo: ASB Classic, Enlace: https://www.wtatennis.com/tournaments/1049/auckland/2026
Torneo: Adelaide International, Enlace: https://www.wtatennis.com/tournaments/2014/adelaide/2026
Torneo: Hobart International, Enlace: https://www.wtatennis.com/tournaments/1050/hobart/2026
Torneo: Australian Open, Enlace: https://www.wtatennis.com/tournaments/australian-open
Torneo: Mubadala Abu Dhabi Open presented by Abu Dhabi Sports Council, Enlace: https://www.wtatennis.com/tournaments/2088/abu-dhabi/2026
Torneo: Ostrava Open, Enlace: https://www.wtatennis.com/tournaments/1154/ostrava/2026
Torneo: Transylvania Open powered by Kaufland, Enlace: https://www.wtatennis.com/tournaments/2050/cluj-napoca/2026
Torneo: Qatar TotalEnergies Open 2026, Enlace: https://www.wtatennis.com/tournaments/doha
Torneo: Dubai Duty Free Tennis Champ

In [ ]:
torneoDatos = {}

## Diccionario de diccionarios
Vamos a actualizar el diccionario, para que le value, en vez de ser unicamente el link, tenga: link, ciudad, jugadoras, categoria... todos los datos que podamos sacar... 

Las jugadoras del torneo me gustaria que a su vez fuese tambien otro diccionario donde tenga el nombre y el enlace a la ficha de esta jugadora

In [ ]:
excepciones = ["United Cup", "Australian Open", "Roland Garros", "The Championships, Wimbledon", "US Open", "WTA Finals Indian Wells"]

for torneo, enlace in tournaments.items():
    print(f"Torneo: {torneo}, Enlace: {enlace}")
    if torneo in excepciones:
        pass
    else:
        torneoDatos[torneo] = {}
        driver = abrir_Web_WTA(enlace)

        ciudad = driver.find_element(By.CLASS_NAME, "tournament-header__description")
        ciudadTexto = ciudad.text
        torneoDatos[torneo]["Ciudad"] = ciudadTexto

        categoria = driver.find_element(By.CLASS_NAME, "tournament-header__tournament-level")
        categoriaTexto = categoria.get_attribute("alt")
        torneoDatos[torneo]["Categoria"] = categoriaTexto

        superficie = driver.find_element(By.CLASS_NAME, "tournament-header__details-surface")
        superficieTexto = superficie.text
        torneoDatos[torneo]["Superficie"] = superficieTexto

        fecha = driver.find_element(By.CSS_SELECTOR, "div.tournament-header__date-range")
        fechaTexto = fecha.text
        torneoDatos[torneo]["Fecha"] = fechaTexto

        botonPlayers = driver.find_element(By.CSS_SELECTOR, "a[data-tracking-detail='Player List']")
        botonPlayers.click()

        jugadoras = driver.find_elements(By.CLASS_NAME, "tournament-player-card__name")
        torneoDatos[torneo]["Jugadoras"] = {}
        for jugadora in jugadoras:
            enlaceJugadora = jugadora.get_attribute("href")
            nombreJgadora = jugadora.get_attribute("data-tracking-player-name")
            torneoDatos[torneo]["Jugadoras"][nombreJgadora] = enlaceJugadora

        driver.close()

Torneo: United Cup, Enlace: https://www.wtatennis.com/tournaments/united-cup
Torneo: Brisbane International, Enlace: https://www.wtatennis.com/tournaments/800/brisbane/2026
✅ Popup 'weblayer--box-web-push-subscription' cerrado exitosamente.
✅ Popup 'weblayer--box-cookieless' cerrado exitosamente.
Torneo: ASB Classic, Enlace: https://www.wtatennis.com/tournaments/1049/auckland/2026
✅ Popup 'weblayer--box-web-push-subscription' cerrado exitosamente.
✅ Popup 'weblayer--box-cookieless' cerrado exitosamente.
Torneo: Adelaide International, Enlace: https://www.wtatennis.com/tournaments/2014/adelaide/2026
✅ Popup 'weblayer--box-web-push-subscription' cerrado exitosamente.
✅ Popup 'weblayer--box-cookieless' cerrado exitosamente.
Torneo: Hobart International, Enlace: https://www.wtatennis.com/tournaments/1050/hobart/2026
✅ Popup 'weblayer--box-web-push-subscription' cerrado exitosamente.
✅ Popup 'weblayer--box-cookieless' cerrado exitosamente.
Torneo: Australian Open, Enlace: https://www.wtaten

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	(No symbol) [0x7ff7c534e745]
	(No symbol) [0x7ff7c4f03d00]
	(No symbol) [0x7ff7c4d33d36]
	(No symbol) [0x7ff7c4d80563]
	(No symbol) [0x7ff7c4db89e2]
	(No symbol) [0x7ff7c4db27a2]
	(No symbol) [0x7ff7c4db1f23]
	(No symbol) [0x7ff7c4cfc8eb]
	(No symbol) [0x7ff7c4cfcf52]
	(No symbol) [0x7ff7c4cfd585]
	(No symbol) [0x7ff7c53f17b1]
	(No symbol) [0x7ff7c54287c7]
	(No symbol) [0x7ff7c541c85e]
	(No symbol) [0x7ff7c50c884e]
	(No symbol) [0x7ff7c4f10415]
	(No symbol) [0x7ff7c4cfbd1e]
	(No symbol) [0x7ff7c54a4cff]
	(No symbol) [0x7fff9c4fdda7]
	(No symbol) [0x7fff9e51ee4c]


In [36]:
import json

# Guardar el diccionario en un archivo
with open("datos.json", "w") as archivo:
    json.dump(torneoDatos, archivo)

In [37]:
# Cargar el diccionario (ejecuta esto al abrir tu notebook)
with open("datos.json", "r") as archivo:
    diccionario_cargado = json.load(archivo)